<a href="https://colab.research.google.com/github/jaegertros/AnatomyLocked-Character-Studio/blob/main/OtherSetups/comfyui_colab_with_manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!python -m pip install -q jedi
# 1. Update the environment to the bleeding edge
# 1. Install PyTorch Nightly with CUDA 13.0 support
#!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130 --force-reinstall

# 2. Fix the Transformers 'ImportError' (Essential for Wan 2.1 Text Encoders)
!pip install -U transformers accelerate

# 3. Install missing Node dependencies that are failing in your logs
!pip install onnxruntime-gpu insightface piexif sageattention pyhocon onnx

# 4. Clear the 'py.nodes' / Lora-Manager pathing issue
# This forces the Python path to recognize the ComfyUI directory correctly
import sys
import os
sys.path.append('/content/drive/MyDrive/AI/ComfyUI')
import os

In [3]:
# 1. Set environment variables
import os
os.environ["NUMEXPR_MAX_THREADS"] = "48"
os.environ["CUDA_MODULE_LOADING"] = "LAZY" # Faster startup on H100
`COMFYUI_MODEL_PATH` environment variable is not set. Assuming `/content/drive/MyDrive/AI/ComfyUI/models`
# 2. Install the acceleration stack
!pip install triton sageattention

# 3. Verify CUDA 13.0 and GPU visibility
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

PyTorch Version: 2.12.0.dev20260225+cu130
CUDA Available: True
Device: NVIDIA L4


In [4]:
#@title Environment Setup

from pathlib import Path

OPTIONS = {}

COMFYUI_MODEL_PATH = "/content/drive/MyDrive/AI"  #@param {type:"string"}

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True  #@param {type:"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}
AI_ROOT = "/content/drive/MyDrive/AI"  #@param {type:"string"}
MODELS_ROOT = "/content/drive/MyDrive/AI/models"  #@param {type:"string"}
GENERATE_MODEL_INVENTORY = True  #@param {type:"boolean"}
INVENTORY_LIMIT = 10  #@param {type:"integer"}

OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

AI_ROOT = AI_ROOT.rstrip('/')
if not MODELS_ROOT.strip():
    MODELS_ROOT = f"{AI_ROOT}/models"
MODELS_ROOT = MODELS_ROOT.rstrip('/')
IMAGES_ROOT = f"{AI_ROOT}/Images"
COMFY_OUTPUT_DIR = IMAGES_ROOT

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /
    from google.colab import drive
    drive.mount('/content/drive')

# Ensure workspace roots exist even on first run.
!mkdir -p "{AI_ROOT}"
!mkdir -p "{MODELS_ROOT}"
!mkdir -p "{IMAGES_ROOT}"

import yaml

WORKSPACE = f"{AI_ROOT}/ComfyUI"

%cd "{AI_ROOT}"
![ ! -d "{WORKSPACE}" ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI "{WORKSPACE}"
%cd "{WORKSPACE}"

if OPTIONS['UPDATE_COMFY_UI']:
    !echo -= Updating ComfyUI =-

    # Force official upstream to avoid stale forks in persistent Drive workspaces.
    !git config --global --add safe.directory "{WORKSPACE}"
    !git remote set-url origin https://github.com/comfyanonymous/ComfyUI
    !git fetch origin
    !git checkout -B master origin/master
    !git reset --hard origin/master

    # Keep execute bits on Drive-backed files where needed.
    ![ -f ".ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/nightly/windows_base_files/run_nvidia_gpu.bat" ] && chmod 755 .ci/nightly/windows_base_files/run_nvidia_gpu.bat
    ![ -f ".ci/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/update_windows/update.py" ] && chmod 755 .ci/update_windows/update.py
    ![ -f ".ci/update_windows/update_comfyui.bat" ] && chmod 755 .ci/update_windows/update_comfyui.bat
    ![ -f ".ci/update_windows/README_VERY_IMPORTANT.txt" ] && chmod 755 .ci/update_windows/README_VERY_IMPORTANT.txt
    ![ -f ".ci/update_windows/run_cpu.bat" ] && chmod 755 .ci/update_windows/run_cpu.bat
    ![ -f ".ci/update_windows/run_nvidia_gpu.bat" ] && chmod 755 .ci/update_windows/run_nvidia_gpu.bat

    !git pull --ff-only

!echo -= Sync Python dependencies =-
!python -m pip install -q -U pip setuptools wheel
!python -m pip install -q -r requirements.txt
!python -m pip install -q accelerate
!python -m pip install -q einops "transformers>=4.28.1" "safetensors>=0.4.2" aiohttp pyyaml Pillow scipy tqdm psutil "tokenizers>=0.13.3"
!python -m pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!python -m pip install -q torchsde
!python -m pip install -q "kornia>=0.7.1" spandrel soundfile sentencepiece
# Frontend package fallback for environments missing bundled frontend assets.
!python -m pip install -q comfyui-frontend-package || true

if OPTIONS['USE_COMFYUI_MANAGER']:
    %cd "{WORKSPACE}/custom_nodes"

    # Keep execute bits on Drive-backed files where needed.
    ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
    ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
    ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
    ![ -f "ComfyUI-Manager/node_db/tutorial/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/tutorial/scan.sh
    ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
    ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat

    ![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager
    %cd ComfyUI-Manager
    !git pull --ff-only

%cd "{WORKSPACE}"

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] and Path("custom_nodes/ComfyUI-Manager/cm-cli.py").exists():
    !echo -= Install custom nodes dependencies =-
    !python -m pip install -q GitPython
    !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies

MODEL_DIRS = {
    "models_root": MODELS_ROOT,
    "diffusion_base": f"{MODELS_ROOT}/diffusion_base",
    "diffusion_sd": f"{MODELS_ROOT}/diffusion_base/sd",
    "diffusion_sdxl": f"{MODELS_ROOT}/diffusion_base/sdxl",
    "checkpoints": f"{MODELS_ROOT}/checkpoints",
    "controlnet": f"{MODELS_ROOT}/controlnet",
    "loras": f"{MODELS_ROOT}/loras",
    "vae": f"{MODELS_ROOT}/vae",
    "clip": f"{MODELS_ROOT}/clip",
    "clip_vision": f"{MODELS_ROOT}/clip_vision",
    "configs": f"{MODELS_ROOT}/configs",
    "embeddings": f"{MODELS_ROOT}/embeddings",
    "upscale_models": f"{MODELS_ROOT}/upscale_models",
    "style_models": f"{MODELS_ROOT}/style_models",
    "gligen": f"{MODELS_ROOT}/gligen",
    "llm": f"{MODELS_ROOT}/llm",
    "audio_models": f"{MODELS_ROOT}/audio_models",
}

for _path in MODEL_DIRS.values():
    Path(_path).mkdir(parents=True, exist_ok=True)

Path(COMFY_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

extra_model_paths = {
    "comfyui": {
        "base_path": MODELS_ROOT,
        "checkpoints": "checkpoints",
        "clip": "clip",
        "clip_vision": "clip_vision",
        "configs": "configs",
        "controlnet": "controlnet",
        "embeddings": "embeddings",
        "loras": "loras",
        "upscale_models": "upscale_models",
        "vae": "vae",
        "llm": "llm",
        "audio_models": "audio_models",
    },
    "diffusion_base_sd": {
        "base_path": MODEL_DIRS["diffusion_base"],
        "checkpoints": "sd",
    },
    "diffusion_base_sdxl": {
        "base_path": MODEL_DIRS["diffusion_base"],
        "checkpoints": "sdxl",
    },
    "sd_inpainting": {
        "base_path": AI_ROOT,
        "checkpoints": "stable-diffusion-inpainting-model",
    },
    "sdxl_inpainting": {
        "base_path": AI_ROOT,
        "checkpoints": "sdxl-inpainting-model",
    },
}

with open(f"{WORKSPACE}/extra_model_paths.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(extra_model_paths, f, default_flow_style=False, sort_keys=False)

print(f"Wrote extra model paths: {WORKSPACE}/extra_model_paths.yaml")

MODEL_EXTS = {".ckpt", ".safetensors", ".pt", ".pth", ".bin", ".gguf"}


def _classify_model(path: Path) -> str:
    name = path.name.lower()
    parent_hint = "/".join(part.lower() for part in path.parts)

    if "/diffusion_base/sdxl/" in parent_hint:
        return "sdxl"
    if "/diffusion_base/sd/" in parent_hint:
        return "sd"
    if "/controlnet/" in parent_hint or any(k in name for k in ("controlnet", "openpose", "depth", "normal")):
        return "controlnet"
    if "/loras/" in parent_hint or "lora" in name or "lyco" in name:
        return "lora"
    if name.endswith((".ckpt", ".safetensors", ".pt", ".pth")):
        return "checkpoint"
    return "other"


def _scan_model_registry():
    registry = defaultdict(list)
    roots = {
        "diffusion_sd": Path(MODEL_DIRS["diffusion_sd"]),
        "diffusion_sdxl": Path(MODEL_DIRS["diffusion_sdxl"]),
        "checkpoints": Path(MODEL_DIRS["checkpoints"]),
        "controlnet": Path(MODEL_DIRS["controlnet"]),
        "loras": Path(MODEL_DIRS["loras"]),
    }

    for _root_name, root in roots.items():
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if p.is_file() and p.suffix.lower() in MODEL_EXTS:
                kind = _classify_model(p)
                registry[kind].append(p)
    return registry


def generate_model_inventory(limit=10, write_snapshot=True):
    registry = _scan_model_registry()

    print("Model inventory summary:")
    for key in ("sd", "sdxl", "controlnet", "lora", "checkpoint", "other"):
        entries = registry.get(key, [])
        print(f" - {key:10s}: {len(entries)}")

    lines = ["| Type | Count | Samples |", "| --- | --- | --- |"]
    for key in ("sd", "sdxl", "controlnet", "lora", "checkpoint", "other"):
        entries = registry.get(key, [])
        sample_names = ", ".join(p.name for p in entries[:limit]) if entries else "None"
        if len(entries) > limit:
            sample_names += f" (+{len(entries) - limit} more)"
        lines.append(f"| {key} | {len(entries)} | {sample_names} |")

    snapshot_text = "\n".join(lines)
    if write_snapshot:
        snapshot_path = Path(WORKSPACE) / "model_registry_snapshot.md"
        snapshot_path.write_text(snapshot_text, encoding="utf-8")
        print(f"Wrote model inventory snapshot: {snapshot_path}")

    return registry


def run_comfyui_preflight():
    critical = []
    warnings = []

    ai_root_path = Path(AI_ROOT)
    workspace_path = Path(WORKSPACE)

    if not ai_root_path.exists():
        critical.append(f"AI root is missing: {ai_root_path}")
    if not workspace_path.exists():
        critical.append(f"ComfyUI workspace is missing: {workspace_path}")

    for key, raw_path in MODEL_DIRS.items():
        p = Path(raw_path)
        if not p.exists():
            warnings.append(f"Model path missing ({key}): {p}")

    registry = _scan_model_registry()
    base_count = len(registry.get("sd", [])) + len(registry.get("sdxl", [])) + len(registry.get("checkpoint", []))
    if base_count == 0:
        warnings.append(
            "No base checkpoints found under diffusion_base/sd, diffusion_base/sdxl, or checkpoints."
        )

    if critical:
        print("ComfyUI preflight failed:")
        for msg in critical:
            print(f" - {msg}")
        raise RuntimeError("Critical preflight checks failed. Fix the paths above and re-run.")

    print("ComfyUI preflight passed.")
    if warnings:
        print("Warnings:")
        for msg in warnings:
            print(f" - {msg}")

    return {"warnings": warnings, "registry": registry}


print(f"AI_ROOT: {AI_ROOT}")
print(f"MODELS_ROOT: {MODELS_ROOT}")
print(f"COMFY_OUTPUT_DIR: {COMFY_OUTPUT_DIR}")

Mounting Google Drive...
/
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/AI
/content/drive/MyDrive/AI/ComfyUI
-= Updating ComfyUI =-
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (1/1), 851 bytes | 65.00 KiB/s, done.
From https://github.com/comfyanonymous/ComfyUI
   0467f690..eb8737d6  master     -> origin/master
M	.ci/update_windows/update.py
M	.ci/update_windows/update_comfyui.bat
M	.ci/update_windows/update_comfyui_stable.bat
M	.ci/windows_amd_base_files/README_VERY_IMPORTANT.txt
M	.ci/windows_amd_base_files/run_amd_gpu.bat
M	.ci/windows_amd_base_files/run_amd_gpu_disable_smart_memory.bat
M	.ci/windows_nvidia_base_files/README_VERY_IMPORTANT.txt
M	.ci/windows_nvidia_base_files/run_cpu.bat
M	.ci/windows_nvidia_base_files/run_nvidia_gpu.bat
M	comfy/audi

In [5]:
from collections import defaultdict

# #@title Model Inventory Snapshot

if "generate_model_inventory" not in globals():
    raise RuntimeError("Run the Environment Setup cell first.")

if GENERATE_MODEL_INVENTORY:
    _ = generate_model_inventory(limit=INVENTORY_LIMIT, write_snapshot=True)
else:
    print("GENERATE_MODEL_INVENTORY is False. Skipping inventory generation.")


Model inventory summary:
 - sd        : 2
 - sdxl      : 1
 - controlnet: 7
 - lora      : 1
 - checkpoint: 12
 - other     : 0
Wrote model inventory snapshot: /content/drive/MyDrive/AI/ComfyUI/model_registry_snapshot.md


Download some models/checkpoints/vae or custom comfyui nodes (uncomment the commands for the ones you want)

In [6]:
#@title Checkpoints
import os

def download(url, path, filename=None):
    if filename:
        dest = os.path.join(path, filename)
    else:
        dest = os.path.join(path, os.path.basename(url))

    if not os.path.exists(dest):
        print(f"Downloading {url} to {dest}...")
        if filename:
             !wget -c {url} -O {dest}
        else:
             !wget -c {url} -P {path}
    else:
        print(f"File {dest} already exists, skipping download.")

# Define paths based on your setup
if "MODELS_ROOT" not in globals():
    AI_ROOT = "/content/drive/MyDrive/AI"
    MODELS_ROOT = f"{AI_ROOT}/models"

base_models = MODELS_ROOT.rstrip("/")
ckpt_path = os.path.join(base_models, "checkpoints")
vae_path = os.path.join(base_models, "vae")
lora_path = os.path.join(base_models, "loras")
controlnet_path = os.path.join(base_models, "controlnet")
style_path = os.path.join(base_models, "style_models")
clip_vision_path = os.path.join(base_models, "clip_vision")
gligen_path = os.path.join(base_models, "gligen")
upscale_path = os.path.join(base_models, "upscale_models")

for p in [ckpt_path, vae_path, lora_path, controlnet_path, style_path, clip_vision_path, gligen_path, upscale_path]:
    os.makedirs(p, exist_ok=True)

### SDXL
### I recommend these workflow examples: https://comfyanonymous.github.io/ComfyUI_examples/sdxl/

#download("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors", ckpt_path)
#download("https://huggingface.co/stabilityai/stable-diffusion-xl-refiner-1.0/resolve/main/sd_xl_refiner_1.0.safetensors", ckpt_path)

# SDXL ReVision
download("https://huggingface.co/comfyanonymous/clip_vision_g/resolve/main/clip_vision_g.safetensors", clip_vision_path)

# SD1.5
download("https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt", ckpt_path)

# SD2
download("https://huggingface.co/stabilityai/stable-diffusion-2-1-base/resolve/main/v2-1_512-ema-pruned.safetensors", ckpt_path)
download("https://huggingface.co/stabilityai/stable-diffusion-2-1/resolve/main/v2-1_768-ema-pruned.safetensors", ckpt_path)

# Some SD1.5 anime style
#download("https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors", ckpt_path)
#download("https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AOM3A1_orangemixs.safetensors", ckpt_path)
#download("https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix3/AOM3A3_orangemixs.safetensors", ckpt_path)
#download("https://huggingface.co/Linaqruf/anything-v3.0/resolve/main/anything-v3-fp16-pruned.safetensors", ckpt_path)

# Waifu Diffusion 1.5 (anime style SD2.x 768-v)
#download("https://huggingface.co/waifu-diffusion/wd-1-5-beta3/resolve/main/wd-illusion-fp16.safetensors", ckpt_path)


# unCLIP models
download("https://huggingface.co/comfyanonymous/illuminatiDiffusionV1_v11_unCLIP/resolve/main/illuminatiDiffusionV1_v11-unclip-h-fp16.safetensors", ckpt_path)
download("https://huggingface.co/comfyanonymous/wd-1.5-beta2_unCLIP/resolve/main/wd-1-5-beta2-aesthetic-unclip-h-fp16.safetensors", ckpt_path)


# VAE
download("https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors", vae_path)
#download("https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/VAEs/orangemix.vae.pt", vae_path)
#download("https://huggingface.co/hakurei/waifu-diffusion-v1-4/resolve/main/vae/kl-f8-anime2.ckpt", vae_path)


# Loras
download("https://civitai.com/api/download/models/10350", lora_path, "theovercomer8sContrastFix_sd21768.safetensors") #theovercomer8sContrastFix SD2.x 768-v
download("https://civitai.com/api/download/models/10638", lora_path, "theovercomer8sContrastFix_sd15.safetensors") #theovercomer8sContrastFix SD1.x
download("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_offset_example-lora_1.0.safetensors", lora_path) #SDXL offset noise lora


# T2I-Adapter
download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_depth_sd14v1.pth", controlnet_path)
download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_seg_sd14v1.pth", controlnet_path)
download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_sketch_sd14v1.pth", controlnet_path)
download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_keypose_sd14v1.pth", controlnet_path)
download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_openpose_sd14v1.pth", controlnet_path)
download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_color_sd14v1.pth", controlnet_path)
download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_canny_sd14v1.pth", controlnet_path)

# T2I Styles Model
download("https://huggingface.co/TencentARC/T2I-Adapter/resolve/main/models/t2iadapter_style_sd14v1.pth", style_path)

# CLIPVision model (needed for styles model)
#ownload("https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/pytorch_model.bin", clip_vision_path, "clip_vit14.bin")


# ControlNet
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11e_sd15_ip2p_fp16.safetensors", controlnet_path)
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11e_sd15_shuffle_fp16.safetensors", controlnet_path)
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_canny_fp16.safetensors", controlnet_path)
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11f1p_sd15_depth_fp16.safetensors", controlnet_path)
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_inpaint_fp16.safetensors", controlnet_path)
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_lineart_fp16.safetensors", controlnet_path)
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_mlsd_fp16.safetensors", controlnet_path)
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_normalbae_fp16.safetensors", controlnet_path)
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_openpose_fp16.safetensors", controlnet_path)
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_scribble_fp16.safetensors", controlnet_path)
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_seg_fp16.safetensors", controlnet_path)
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_softedge_fp16.safetensors", controlnet_path)
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15s2_lineart_anime_fp16.safetensors", controlnet_path)
download("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11u_sd15_tile_fp16.safetensors", controlnet_path)

# ControlNet SDXL
download("https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-canny-rank256.safetensors", controlnet_path)
download("https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-depth-rank256.safetensors", controlnet_path)
download("https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-recolor-rank256.safetensors", controlnet_path)
download("https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-sketch-rank256.safetensors", controlnet_path)

# Controlnet Preprocessor nodes by Fannovel16
!cd custom_nodes && git clone https://github.com/Fannovel16/comfy_controlnet_preprocessors; cd comfy_controlnet_preprocessors && python install.py


# GLIGEN
download("https://huggingface.co/comfyanonymous/GLIGEN_pruned_safetensors/resolve/main/gligen_sd14_textbox_pruned_fp16.safetensors", gligen_path)


# ESRGAN upscale model
download("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth", upscale_path)
download("https://huggingface.co/sberbank-ai/Real-ESRGAN/resolve/main/RealESRGAN_x2.pth", upscale_path)
download("https://huggingface.co/sberbank-ai/Real-ESRGAN/resolve/main/RealESRGAN_x4.pth", upscale_path)


File /content/drive/MyDrive/AI/models/clip_vision/clip_vision_g.safetensors already exists, skipping download.
File /content/drive/MyDrive/AI/models/checkpoints/v1-5-pruned-emaonly.ckpt already exists, skipping download.
--2026-02-26 02:51:06--  https://huggingface.co/stabilityai/stable-diffusion-2-1-base/resolve/main/v2-1_512-ema-pruned.safetensors
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.17, 18.164.174.23, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized

Username/Password Authentication Failed.
--2026-02-26 02:51:06--  https://huggingface.co/stabilityai/stable-diffusion-2-1/resolve/main/v2-1_768-ema-pruned.safetensors
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.17, 18.164.174.23, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized

Username/Password A

### Run ComfyUI with cloudflared (Recommended Way)




In [7]:
!pip install av

In [11]:
!pip uninstall -y sqlalchemy

Found existing installation: SQLAlchemy 1.4.54
Uninstalling SQLAlchemy-1.4.54:
  Successfully uninstalled SQLAlchemy-1.4.54


In [12]:
!python -m pip install sqlalchemy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 15.1 MB/s  0:00:00


In [13]:
!python -m pip install sqlalchemy-orm


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for sqlalchemy-orm: filename=sqlalchemy_orm-1.2.10-py3-none-any.whl size=27372 sha256=d2f28a0cb48695950adf2bde483801e37ca4366ce07fd1047cca67ac11691ad1
  Stored in directory: /root/.cache/pip/wheels/d1/36/d4/d0b42a176c4451607cf537133117210438fa0916b5de6b6cf4
Successfully built sqlalchemy-orm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sqlalchemy-orm]


In [ ]:
from collections import defaultdict
!wget -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

from pathlib import Path
import subprocess
import threading
import time
import socket
import urllib.request

if "run_comfyui_preflight" in globals():
  run_comfyui_preflight()

output_dir = globals().get("COMFY_OUTPUT_DIR", globals().get("IMAGES_ROOT", "/content/drive/MyDrive/AI/Images"))
Path(output_dir).mkdir(parents=True, exist_ok=True)

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server --output-directory "{output_dir}"

--2026-02-26 04:04:21--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.2.0/cloudflared-linux-amd64.deb [following]
--2026-02-26 04:04:22--  https://github.com/cloudflare/cloudflared/releases/download/2026.2.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/4fddf4d7-e02d-44dc-9e5a-ef9e28afdd54?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-26T04%3A40%3A15Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

### Run ComfyUI with localtunnel




In [ ]:
!npm install -g localtunnel

from pathlib import Path
import subprocess
import threading
import time
import socket
import urllib.request

if "run_comfyui_preflight" in globals():
  run_comfyui_preflight()

output_dir = globals().get("COMFY_OUTPUT_DIR", globals().get("IMAGES_ROOT", "/content/drive/MyDrive/AI/Images"))
Path(output_dir).mkdir(parents=True, exist_ok=True)

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server --output-directory "{output_dir}"

### Run ComfyUI with colab iframe (use only in case the previous way with localtunnel doesn't work)

You should see the ui appear in an iframe. If you get a 403 error, it's your firefox settings or an extension that's messing things up.

If you want to open it in another window use the link.

Note that some UI features like live image previews won't work because the colab iframe blocks websockets.

In [ ]:
from pathlib import Path
import threading
import time
import socket

if "run_comfyui_preflight" in globals():
  run_comfyui_preflight()

output_dir = globals().get("COMFY_OUTPUT_DIR", globals().get("IMAGES_ROOT", "/content/drive/MyDrive/AI/Images"))
Path(output_dir).mkdir(parents=True, exist_ok=True)
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  from google.colab import output
  output.serve_kernel_port_as_iframe(port, height=1024)
  print("to open it in a window you can open this link here:")
  output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server --output-directory "{output_dir}"